# 🧠 Tutorial #1 — Explore the Data (Interactive)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atifhalim/BrainWerks/blob/main/learning/notebooks/Tutorial1_Explore_Data.ipynb)

Pick a **data source** from the dropdown and see its **raw brain data** as a table —
just like the deck's "raw data" slide. Some sources are **real EEG** from real people;
one is the **synthetic (random)** data from Tutorial #1.

### How to use it
1. In the menu, click **Runtime ▸ Run all** (or press each ▶️ in order).
2. Scroll to the bottom, then use the **Data source ▾** dropdown.
3. The raw data table (and a little plot) appear right below it.

> The first time you pick a *real* source it downloads a small sample, so give it a few seconds.


## Step 1 — Set up (run once)

In [ ]:
# Setup — installs the packages and keeps them compatible with Colab.
import subprocess, sys
print("Setting up… installing packages (~30–60s the first time).")
pkgs = ["mne", "ipywidgets", "pandas<3"]
res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                     capture_output=True, text=True)
# Colab preloads some packages we do NOT use here (e.g. moviepy). pip prints
# harmless "dependency resolver" notes about them — we hide those, but never
# hide a real failure.
IGNORE = ("dependency resolver", "moviepy", "decorator", "google-colab", "pip's ")
def _noise(l):
    return (not l.strip()) or any(k in l for k in IGNORE)
if res.returncode != 0:
    lines = [l for l in (res.stdout + res.stderr).splitlines() if not _noise(l)]
    print("\nInstall problem:\n" + "\n".join(lines[-20:]))
    raise SystemExit("Setup failed. Try Runtime ▸ Restart session, then run again.")
print("✅ Setup complete — you can ignore any Colab package notes.")

## Step 2 — Teach the notebook how to load each source

Each loader returns the signal as a grid of **channels × time**, in microvolts (µV).
You don't need to read every line — just run it.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mne
mne.set_log_level("ERROR")

SECONDS = 4          # how many seconds of signal to show
_cache = {}          # remember downloads so switching sources is fast

def _pack(data_uv, ch, sfreq, name, note, real):
    return dict(data=data_uv, ch=list(ch), sfreq=float(sfreq), name=name, note=note, real=real)

def load_synthetic():
    sfreq, rng = 250.0, np.random.default_rng(7)
    n = int(SECONDS * sfreq)
    data = rng.standard_normal((3, n)) * 8.0        # ~8 µV of pure noise
    return _pack(data, ["C3", "Cz", "C4"], sfreq, "Synthetic (random noise)",
                 "Made-up random data — there is NO real pattern. That is the whole point of Tutorial #1.", False)

def _eegbci(run, picks, name, note):
    fn = mne.datasets.eegbci.load_data(1, [run], update_path=True)     # PhysioNet, subject 1
    raw = mne.io.read_raw_edf(fn[0], preload=True, verbose="ERROR")
    mne.datasets.eegbci.standardize(raw)                              # rename to C3, Cz, ...
    raw.pick(picks)
    sf = raw.info["sfreq"]
    data = raw.get_data(stop=int(SECONDS * sf)) * 1e6                  # volts -> µV
    return _pack(data, raw.ch_names, sf, name, note, True)

def load_motor():
    return _eegbci(4, ["C3", "Cz", "C4"],
        "Real motor imagery (imagine moving a hand)",
        "Real EEG over the motor area while a person imagined moving a fist (PhysioNet EEGBCI).")

def load_alpha():
    return _eegbci(2, ["O1", "Oz", "O2"],
        "Real alpha waves (eyes closed)",
        "Real EEG over the back of the head with eyes CLOSED — the alpha rhythm gets strong (PhysioNet EEGBCI).")

def load_sleep():
    paths = mne.datasets.sleep_physionet.age.fetch_data(subjects=[0], recording=[1], on_missing="warn")
    raw = mne.io.read_raw_edf(paths[0][0], preload=True, verbose="ERROR")
    raw.pick([c for c in raw.ch_names if c.startswith("EEG")])
    sf = raw.info["sfreq"]
    start = int(20 * 60 * sf)                                          # jump ~20 min in (asleep)
    data = raw.get_data(start=start, stop=start + int(SECONDS * sf)) * 1e6
    return _pack(data, raw.ch_names, sf, "Real sleep EEG",
                 "Real EEG recorded while a person was asleep — notice the big slow waves (PhysioNet Sleep-EDF).", True)

LOADERS = {
    "Synthetic (Tutorial #1 random data)": load_synthetic,
    "Real motor imagery (imagine a hand)": load_motor,
    "Real alpha waves (eyes closed)":      load_alpha,
    "Real sleep EEG":                      load_sleep,
}

def get_source(name):
    if name not in _cache:
        _cache[name] = LOADERS[name]()
    return _cache[name]

print("Ready! Sources:", ", ".join(LOADERS))

## Step 3 — Pick a source and see the raw data 👇

Remember the vocabulary from the deck:

* **Each column = one channel** (one electrode on the head).
* **Each row = one moment in time** (one *sample*). At 250 Hz there are 250 rows per second.
* **Each number = a voltage** in microvolts (µV) — the tiny brain signal at that spot and moment.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
try:
    from google.colab import output as _colab_out
    _colab_out.enable_custom_widget_manager()
except Exception:
    pass

def show(source):
    print("Loading… (a real source downloads a small sample the first time)")
    d = get_source(source)
    data, ch, sf = d["data"], d["ch"], d["sfreq"]
    n = data.shape[1]
    t_ms = np.round(np.arange(n) / sf * 1000, 1)

    df = pd.DataFrame(data.T, columns=[f"{c} (µV)" for c in ch]).round(2)
    df.insert(0, "Time (ms)", t_ms)

    tag = "REAL brain data ✅" if d["real"] else "SYNTHETIC (made-up) 🎲"
    print(f"\n=== {d['name']}  —  {tag} ===")
    print(d["note"])
    print(f"channels = {len(ch)}   ·   sampling rate = {sf:.0f} Hz   ·   showing {SECONDS}s = {n} samples")
    print(f"data shape (channels × time) = {data.shape}\n")

    # quick look: first 2 seconds, channels stacked apart so they don't overlap
    show_n = min(n, int(2 * sf))
    plt.figure(figsize=(9, 2.6))
    for i, c in enumerate(ch):
        plt.plot(t_ms[:show_n], data[i, :show_n] + i * 40, lw=0.8, label=c)
    plt.xlabel("time (ms)"); plt.yticks([]); plt.title("first 2 seconds")
    plt.legend(loc="upper right", fontsize=8); plt.tight_layout(); plt.show()

    print("First 15 rows of the raw data:")
    display(df.head(15))

widgets.interact(
    show,
    source=widgets.Dropdown(options=list(LOADERS), description="Data source:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="540px")),
);

## Next step — train a model on this data

Ready to go from *seeing* the data to *training* on it? Open the companion notebook, where you pick a task, pick a model, set the training options, and watch the accuracy:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atifhalim/BrainWerks/blob/main/learning/notebooks/Tutorial1_Train_Model.ipynb)

**➡ Tutorial1_Train_Model.ipynb**
